# Sandbox — smoke-test `lib.py` pieces

Throwaway notebook for poking at functions in `lib.py` without contaminating
the experiment notebooks. Use this to verify the hook returns the right
shape, the tokenizer behaves as expected, MM direction has a sane magnitude,
etc. Cells are independent — run whichever you need.

In [5]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import torch
import lib

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Tokenizer + prompt construction

In [2]:
# Load tokenizer only (no model — fast) to test prompt building.
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(lib.MODEL_PATH)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

p_honest = lib._make_prompt("an honest", "Paris is the capital of")
p_untrue = lib._make_prompt("an untruthful", "Paris is the capital of")
print("Honest prompt:")
print(repr(p_honest))
print()
print("Untruthful prompt (same prefix):")
print(repr(p_untrue))

# Tokenize both — verify last token is the same (it's the prefix end), and
# that left-padding works as expected.
batch = tokenizer([p_honest, p_untrue], padding=True, return_tensors="pt")
print(f"\ninput_ids shape: {batch['input_ids'].shape}")
print(f"Last token of each (decoded): {[tokenizer.decode([t]) for t in batch['input_ids'][:, -1]]}")
print(f"Last 5 tokens of honest:      {tokenizer.convert_ids_to_tokens(batch['input_ids'][0, -5:].tolist())}")
print(f"Last 5 tokens of untruthful:  {tokenizer.convert_ids_to_tokens(batch['input_ids'][1, -5:].tolist())}")

Honest prompt:
"<|im_start|>user\nPretend you're an honest person making statements about the world.<|im_end|>\n<|im_start|>assistant\nParis is the capital of"

Untruthful prompt (same prefix):
"<|im_start|>user\nPretend you're an untruthful person making statements about the world.<|im_end|>\n<|im_start|>assistant\nParis is the capital of"

input_ids shape: torch.Size([2, 29])
Last token of each (decoded): [' of', ' of']
Last 5 tokens of honest:      ['Paris', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof']
Last 5 tokens of untruthful:  ['Paris', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof']


## RepE pair construction on a tiny subset

In [3]:
train, test = lib.build_repe_pairs(
    lib.DATA_DIR / "facts_true_false.csv",
    tokenizer,
    seed=0,
    n_train_pairs=4,   # tiny — just to inspect output shape
    n_test_pairs=2,
)
print(f"Train rows: {len(train)}, Test rows: {len(test)}")
print(f"\nUnique statement_ids in train: {sorted({r['statement_id'] for r in train})}")
print(f"Label counts (train): honest={sum(r['label']==1 for r in train)}, "
      f"untruthful={sum(r['label']==0 for r in train)}")
print(f"\nFirst 4 prompts in train:")
for r in train[:4]:
    print(f"  [label={r['label']}, sid={r['statement_id']}]")
    print(f"    {r['prompt']!r}")

Train rows: 68, Test rows: 18

Unique statement_ids in train: [0, 1, 2, 3]
Label counts (train): honest=34, untruthful=34

First 4 prompts in train:
  [label=1, sid=2]
    "<|im_start|>user\nPretend you're an honest person making statements about the world.<|im_end|>\n<|im_start|>assistant\nThe four fundamental forces of"
  [label=0, sid=2]
    "<|im_start|>user\nPretend you're an untruthful person making statements about the world.<|im_end|>\n<|im_start|>assistant\nThe four fundamental forces of nature are"
  [label=1, sid=3]
    "<|im_start|>user\nPretend you're an honest person making statements about the world.<|im_end|>\n<|im_start|>assistant\nSound travels as"
  [label=0, sid=2]
    "<|im_start|>user\nPretend you're an untruthful person making statements about the world.<|im_end|>\n<|im_start|>assistant\nThe four fundamental forces of nature are gravity, electromagnetism, the strong nuclear force,"


## Model load (slow — only run when you need to test the hook)

In [4]:
model, tok = lib.load_model_and_tokenizer()
print(f"n_layers={model.config.num_hidden_layers}, hidden={model.config.hidden_size}")
print(f"GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

n_layers=36, hidden=2560
GPU memory after load: 8.04 GB


## Smoke test: `collect_activations`

Tiny batch (4 prompts) → verify output shape, dtype, that vectors aren't
all-NaN, and that the same statement under honest vs untruthful gives
*different* activations (sanity: if the two are equal, you're extracting at
a position before the SP took effect, which means your hook is wrong).

In [ ]:
# 2 statements x 2 personas = 4 prompts
prompts = [
    lib._make_prompt("an honest",     "The sun is a"),
    lib._make_prompt("an untruthful", "The sun is a"),
    lib._make_prompt("an honest",     "Water boils at"),
    lib._make_prompt("an untruthful", "Water boils at"),
]
labels = [1, 0, 1, 0]

acts = lib.collect_activations(model, tok, prompts, layer=14)

print(f"acts shape: {acts.shape}")  # expect (4, hidden)
print(f"acts dtype: {acts.dtype}")
print(f"any NaN:    {torch.isnan(acts).any().item()}")
print(f"|acts|:     {acts.float().norm(dim=-1).tolist()}")

# Sanity: same statement, different persona — activations should differ.
diff_sun   = (acts[0] - acts[1]).float().norm().item()
diff_water = (acts[2] - acts[3]).float().norm().item()
print(f"\nhonest vs untruthful |Δ| (sun):   {diff_sun:.3f}")
print(f"honest vs untruthful |Δ| (water): {diff_water:.3f}")
print("Both should be clearly > 0. If ~0, the hook is reading something prompt-shared.")

## Smoke test: `mm_direction`

In [7]:
# Dummy data: two well-separated Gaussians. MM direction should point along
# the separation axis (here: the all-ones direction).
rng = np.random.default_rng(0)
D = 64
X_neg = rng.normal(loc=-1.0, scale=1.0, size=(200, D))
X_pos = rng.normal(loc=+1.0, scale=1.0, size=(200, D))
X = np.concatenate([X_neg, X_pos], axis=0).astype(np.float32)
y = np.array([0]*200 + [1]*200)

d = lib.mm_direction(X, y)
print(f"d shape: {d.shape}")
print(f"d mean coord (expect ~2.0 since means differ by 2 per dim): {d.mean():.3f}")
print(f"d std across coords (expect small, since true direction is uniform): {d.std():.3f}")

# Project and check AUROC — should be near 1.0
scores = lib.mm_score(X, d)
print(f"MM AUROC on toy data: {lib.auroc(scores, y):.4f}")

d shape: (64,)
d mean coord (expect ~2.0 since means differ by 2 per dim): 2.003
d std across coords (expect small, since true direction is uniform): 0.098
MM AUROC on toy data: 1.0000


## Cache I/O smoke test

In [8]:
lib.save_cache("smoke", a=torch.tensor([1, 2, 3]), b=torch.randn(2, 3))
print("exists:", lib.cache_exists("smoke"))
blob = lib.load_cache("smoke")
print(blob)
lib.cache_path("smoke").unlink()

exists: True
{'a': tensor([1, 2, 3]), 'b': tensor([[-1.0973,  1.1886, -0.3139],
        [-0.6836, -0.6783, -0.6821]])}


## Verify your `lib.steering` implementation (Exp 5 gear)

`lib.steering` is a `TODO_PETER` stub. Implement it (register a forward hook on
`model.model.layers[layer]` that adds `coeff*vector` to the block's residual-stream
output, masked to `pos_mask`, removed on exit), then run the three tests below.

- **Test 1** — the hook adds *exactly* `coeff*vector` at the steered position.
- **Test 2** — the hook is removed after the `with` block (no leak into later runs).
- **Test 3** — behavioral: steering ±θ_truth moves the model's True/False logit.

If Test 1 passes you've got the mechanism right; 2 guards the nastiest bug (a hook
that never gets removed silently steers everything afterward); 3 confirms it actually
propagates to the output. Needs the model loaded (run the model-load cell above).

In [27]:
# Needs `model, tok` from the model-load cell above.
try:
    model, tok
except NameError:
    model, tok = lib.load_model_and_tokenizer()

LAYER = 20
H = model.config.hidden_size
test_prompts = [
    lib._make_eval_prompt("Paris is the capital of France."),
    lib._make_eval_prompt("The moon is made of cheese."),
]

# ---- Test 1: hook adds exactly coeff*vector at the steered (last) position ----
base = lib.collect_activations(model, tok, test_prompts, layer=LAYER)   # (2, H), last-token
vec = torch.zeros(H); vec[7] = 10.0
COEFF = 2.0
with lib.steering(model, LAYER, vec, coeff=COEFF):        # pos_mask=None -> all positions
    steered = lib.collect_activations(model, tok, test_prompts, layer=LAYER)
delta = (steered - base).float()
print(f"Test 1: delta[:, 7] = {delta[:, 7].tolist()}  (expect ~{COEFF*10.0})")
print(f"        delta off-target max = {delta[:, :7].abs().max().item():.3f}  (expect ~0)")
assert torch.allclose(delta[:, 7], torch.full((len(test_prompts),), COEFF*10.0), atol=0.7), \
    "Test 1 FAILED — hook is not adding coeff*vector to the L20 residual."
print("Test 1 PASSED: hook adds coeff*vector at the right layer.\n")

# ---- Test 2: hook removed after the context (no lingering steering) ----
after = lib.collect_activations(model, tok, test_prompts, layer=LAYER)
assert torch.allclose(after.float(), base.float(), atol=1e-2), \
    "Test 2 FAILED — activations still perturbed after the `with` block; hook not removed!"
print("Test 2 PASSED: hook cleaned up after the context.\n")

# ---- Test 3: behavioral — steering ±θ_truth moves the True/False logit ----
tb = lib.load_cache("truth_layer20")
Xtr, ytr = tb['X_train_last'].float().numpy(), tb['y_train'].numpy()
d_mm = lib.mm_direction(Xtr, ytr)
v = lib.matched_steering_vector(d_mm, Xtr[ytr == 1].mean(0), Xtr[ytr == 0].mean(0))
ti, fi = lib.truth_token_ids(tok)
fp = [lib._make_eval_prompt("The moon is made of cheese.")]   # a FALSE statement
base_d = lib.truth_logit_diff(model, tok, fp, ti, fi)[0]
pos_d  = lib.truth_logit_diff(model, tok, fp, ti, fi, steer=dict(layer=LAYER, vector=v, coeff=+8.0, mode='context'))[0]
neg_d  = lib.truth_logit_diff(model, tok, fp, ti, fi, steer=dict(layer=LAYER, vector=v, coeff=-8.0, mode='context'))[0]
print(f"Test 3: Δ(logP True - logP False) on a FALSE statement:")
print(f"        baseline = {base_d:+.2f}   +8·θ_mm = {pos_d:+.2f}   -8·θ_mm = {neg_d:+.2f}")
print("        Expect +θ to RAISE Δ (more 'true') and -θ to LOWER it. Flat => hook not affecting the forward.")

Test 1: delta[:, 7] = [20.0, 20.0]  (expect ~20.0)
        delta off-target max = 0.000  (expect ~0)
Test 1 PASSED: hook adds coeff*vector at the right layer.

Test 2 PASSED: hook cleaned up after the context.

Test 3: Δ(logP True - logP False) on a FALSE statement:
        baseline = -24.38   +8·θ_mm = -2.37   -8·θ_mm = -0.77
        Expect +θ to RAISE Δ (more 'true') and -θ to LOWER it. Flat => hook not affecting the forward.


### Test 4 — `pos_mask` steers exactly the selected positions

Tests 1–3 used `pos_mask=None` (all positions). This checks the **masked path** the
NIE readout relies on (`mode='context'`): steer a chosen subset of tokens and verify
the rest — crucially the **final/readout token** — are untouched. We read the L20
block output (`hidden_states[LAYER+1]`) at every position and diff steered vs baseline.

In [28]:
# Needs `model, tok` (model-load cell above). LAYER/H from the Test 1-3 cell.
prompt = [lib._make_eval_prompt("Water boils at 100 degrees Celsius at sea level.")]
inp = tok(prompt, return_tensors="pt").to(model.device)
seq = inp["input_ids"].shape[1]

vec4 = torch.zeros(model.config.hidden_size); vec4[3] = 5.0
# Steer the 2nd half of the tokens, but NOT the final readout token (mimics mode='context').
mask = torch.zeros(1, seq); mask[0, seq // 2:] = 1.0; mask[0, -1] = 0.0

with torch.no_grad():
    base_hs = model(**inp, output_hidden_states=True).hidden_states[LAYER + 1][0]   # (seq, H)
    with lib.steering(model, LAYER, vec4, coeff=1.0, pos_mask=mask):
        steer_hs = model(**inp, output_hidden_states=True).hidden_states[LAYER + 1][0]

delta = (steer_hs - base_hs).float()[:, 3].cpu()   # per-position change on the steered dim (to CPU for asserts)
steered = mask[0].bool().cpu()
print(f"Test 4: {seq} positions; steering {int(steered.sum())} of them (2nd half, minus the last token)")
print(f"        delta on steered positions   (expect ~5): min {delta[steered].min():.2f}, max {delta[steered].max():.2f}")
print(f"        delta on unsteered positions (expect ~0): max |delta| = {delta[~steered].abs().max():.3f}")
assert torch.allclose(delta[steered], torch.full((int(steered.sum()),), 5.0), atol=0.5), \
    "Test 4 FAILED — steered positions did not move by coeff*vector."
assert delta[~steered].abs().max() < 0.5, \
    "Test 4 FAILED — steering leaked into masked-out positions (incl. the readout token!)."
print("Test 4 PASSED: pos_mask steers exactly the selected tokens; the readout token is untouched.")

Test 4: 46 positions; steering 22 of them (2nd half, minus the last token)
        delta on steered positions   (expect ~5): min 4.94, max 5.06
        delta on unsteered positions (expect ~0): max |delta| = 0.000
Test 4 PASSED: pos_mask steers exactly the selected tokens; the readout token is untouched.
